El script debe realizar un agrupamiento o Clustering con AgglomerativeClustering y distancia de Gower

    1. Usar la matriz de distancias de Gower calculada en el Ejercicio 3.

    2. Aplicar AgglomerativeClustering con metric="precomputed" y linkage="average".

    3. Determinar un número adecuado de clusters (Silhouette y/o Davies-Bouldin).

## Paso 1: Calcular la matriz de distancias de Gower

Reutilizamos el código del Ejercicio 3 para calcular la matriz de Gower con una muestra del dataset.


In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# Cargar dataset
df = cargar_dataset_adult()

# Tomar muestra (5000 registros para eficiencia computacional)
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42).copy()

print("="*60)
print("CÁLCULO DE MATRIZ DE DISTANCIAS DE GOWER")
print("="*60)
print(f"\nMuestra utilizada: {sample_size} observaciones")

# Identificar tipos de variables (sin fnlwgt)
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# Manejo de la Variable Ordinal 'education'
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}

# Cálculo Vectorizado de la Matriz de Distancias de Gower
print("\nCalculando matriz de distancias de Gower...")

# Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)

# Asegurar que la diagonal sea exactamente 0 (distancia de un punto consigo mismo)
np.fill_diagonal(gower_matrix, 0)

print(f"\n✓ Matriz de Gower calculada: {gower_matrix.shape}")
print(f"  Rango de distancias: [{gower_matrix.min():.4f}, {gower_matrix.max():.4f}]")
print(f"  Distancia media: {gower_matrix.mean():.4f}")


CÁLCULO DE MATRIZ DE DISTANCIAS DE GOWER

Muestra utilizada: 5000 observaciones

Calculando matriz de distancias de Gower...

✓ Matriz de Gower calculada: (5000, 5000)
  Rango de distancias: [0.0000, 0.9128]
  Distancia media: 0.3295


## Paso 2: Aplicar AgglomerativeClustering con distintos valores de k

Usamos la matriz de distancias precalculada con `metric="precomputed"` y `linkage="average"`.


In [ ]:
# Rango de k a probar
k_range = range(2, 11)

# Listas para almacenar resultados
silhouette_scores = []
davies_bouldin_scores = []
clustering_models = {}

print("="*60)
print("APLICANDO AGGLOMERATIVE CLUSTERING CON DISTINTOS K")
print("="*60)
print("\nConfiguración: metric='precomputed', linkage='average'")
print("\nEntrenando modelos...\n")

# Aplicar AgglomerativeClustering para cada valor de k
for k in k_range:
    # Entrenar AgglomerativeClustering
    # Nota: En scikit-learn >= 1.4, usar 'metric' en lugar de 'affinity'
    agg = AgglomerativeClustering(
        n_clusters=k,
        metric='precomputed',
        linkage='average'
    )
    labels = agg.fit_predict(gower_matrix)
    
    # Guardar el modelo
    clustering_models[k] = {'model': agg, 'labels': labels}
    
    # Calcular métricas
    # IMPORTANTE: Para Silhouette necesitamos la matriz de distancias
    silhouette = silhouette_score(gower_matrix, labels, metric='precomputed')
    
    # Para Davies-Bouldin necesitamos las características originales (solo numéricas)
    df_temp = df.sample(n=sample_size, random_state=42)[numeric_cols]
    davies_bouldin = davies_bouldin_score(df_temp, labels)
    
    # Almacenar resultados
    silhouette_scores.append(silhouette)
    davies_bouldin_scores.append(davies_bouldin)
    
    print(f"k={k:2d} | Silhouette: {silhouette:.4f} | Davies-Bouldin: {davies_bouldin:.4f}")

print("\n✓ Entrenamiento completado para todos los valores de k")


APLICANDO AGGLOMERATIVE CLUSTERING CON DISTINTOS K

Configuración: affinity='precomputed', linkage='average'

Entrenando modelos...



TypeError: AgglomerativeClustering.__init__() got an unexpected keyword argument 'affinity'

## Paso 3: Determinar el número óptimo de clusters

Usamos dos métricas para determinar el k óptimo:

1. **Coeficiente de Silhouette**: Maximizar (valores cercanos a 1 son mejores)
2. **Índice Davies-Bouldin**: Minimizar (valores menores indican mejores clusters)


In [ ]:
# Determinar k óptimos según cada métrica
max_silhouette_k = k_range[np.argmax(silhouette_scores)]
max_silhouette = max(silhouette_scores)

min_db_k = k_range[np.argmin(davies_bouldin_scores)]
min_db = min(davies_bouldin_scores)

# Resumen de resultados
print("="*60)
print("DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLUSTERS")
print("="*60)
print(f"\nSegún Coeficiente de Silhouette:")
print(f"   k óptimo = {max_silhouette_k} (Silhouette = {max_silhouette:.4f})")

print(f"\nSegún Índice Davies-Bouldin:")
print(f"   k óptimo = {min_db_k} (Davies-Bouldin = {min_db:.4f})")

# Tabla resumen
results_df = pd.DataFrame({
    'k': list(k_range),
    'Silhouette': silhouette_scores,
    'Davies-Bouldin': davies_bouldin_scores
})

print(f"\n\nTabla Resumen de Métricas:")
print(results_df.to_string(index=False))


## Paso 4: Análisis del modelo óptimo

Analizamos los clusters usando el k óptimo determinado por las métricas.


In [ ]:
# Usar el k óptimo según Silhouette (puedes cambiar a min_db_k si prefieres Davies-Bouldin)
k_optimal = max_silhouette_k
labels_optimal = clustering_models[k_optimal]['labels']

# Recuperar datos originales (antes de transformar education)
df_original = df.sample(n=sample_size, random_state=42)
all_cols = numeric_cols + categorical_cols
df_clustered = df_original[all_cols].copy()
df_clustered['Cluster'] = labels_optimal

print("="*60)
print(f"ANÁLISIS DEL MODELO ÓPTIMO (k={k_optimal})")
print("="*60)

# Estadísticas por cluster
print(f"\nDistribución de observaciones por cluster:")
cluster_counts = pd.Series(labels_optimal).value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(labels_optimal)) * 100
    print(f"   Cluster {cluster_id}: {count:6d} observaciones ({percentage:5.2f}%)")

# Características promedio por cluster (solo numéricas)
print(f"\n\nCaracterísticas numéricas promedio por cluster:\n")
cluster_summary = df_clustered.groupby('Cluster')[numeric_cols].mean()
print(cluster_summary.round(2))

# Distribución de variables categóricas importantes por cluster
print(f"\n\nCaracterísticas categóricas más frecuentes por cluster:")
for cluster_id in sorted(cluster_summary.index):
    print(f"\n   CLUSTER {cluster_id}:")
    cluster_data = df_clustered[df_clustered['Cluster'] == cluster_id]
    
    # Variables categóricas clave
    for cat_col in ['workclass', 'education', 'marital-status', 'income']:
        if cat_col in df_clustered.columns:
            mode_val = cluster_data[cat_col].mode()[0] if len(cluster_data[cat_col].mode()) > 0 else 'N/A'
            freq = (cluster_data[cat_col] == mode_val).sum() / len(cluster_data) * 100
            print(f"   • {cat_col}: {mode_val} ({freq:.1f}%)")

print(f"\n\n✓ Análisis completado con k óptimo = {k_optimal}")


## Paso 5: Interpretación detallada de características por cluster

Comparación de las características promedio de cada cluster con la media global.


In [ ]:
# Interpretación detallada
print("="*60)
print("INTERPRETACIÓN DETALLADA DE CLUSTERS")
print("="*60)

for cluster_id in sorted(cluster_summary.index):
    print(f"\n🔹 CLUSTER {cluster_id}:")
    print(f"   Tamaño: {cluster_counts[cluster_id]} observaciones ({cluster_counts[cluster_id]/len(labels_optimal)*100:.1f}%)")
    
    print(f"\n   Características numéricas:")
    for col in numeric_cols:
        val = cluster_summary.loc[cluster_id, col]
        global_mean = df_original[col].mean()
        diff_percent = ((val - global_mean) / global_mean * 100) if global_mean != 0 else 0
        
        if abs(diff_percent) > 20:  # Diferencia significativa
            direction = "↑" if diff_percent > 0 else "↓"
            print(f"   • {col}: {val:.1f} ({direction} {abs(diff_percent):.0f}% vs media global)")
        else:
            print(f"   • {col}: {val:.1f} (~ media global)")
    
    print(f"\n   Características categóricas dominantes:")
    cluster_data = df_clustered[df_clustered['Cluster'] == cluster_id]
    for cat_col in ['education', 'marital-status', 'income']:
        if cat_col in df_clustered.columns:
            mode_val = cluster_data[cat_col].mode()[0] if len(cluster_data[cat_col].mode()) > 0 else 'N/A'
            freq = (cluster_data[cat_col] == mode_val).sum() / len(cluster_data) * 100
            print(f"   • {cat_col}: {mode_val} ({freq:.1f}%)")

print("\n" + "="*60)
print("="*60)
